# Synthetic mammalian fed-batch: mechanistic → surrogate

No experimental accuracy claim. Inputs supply all model and operating choices.


In [ ]:
from pathlib import Path
import json, sys, itertools
from copy import deepcopy
from types import SimpleNamespace
import numpy as np
import matplotlib.pyplot as plt
from PharmaPy.Bioreactors import build_bioreactor, register_rate_provider
EXAMPLE = Path.cwd()
if not (EXAMPLE / 'inputs/case.json').exists():
    EXAMPLE = EXAMPLE / 'examples/bioreactors/mammalian_surrogate_fed_batch'
EXPORT_DIR = EXAMPLE / 'outputs'
sys.path.insert(0, str(EXAMPLE))
from polynomial_provider import PolynomialProvider


In [ ]:
case = json.loads((EXAMPLE/'inputs/case.json').read_text())
mechanism = json.loads((EXAMPLE/'inputs/mechanism.json').read_text())
settings = json.loads((EXAMPLE/'inputs/surrogate.json').read_text())
thermo = EXAMPLE/'inputs/thermo.json'
baseline = build_bioreactor(case, mechanism, thermo)
histories = {'mechanistic': baseline.solve(verbose=False)}
outputs = tuple(dict.fromkeys(mechanism['model']['kinetics']['species_mass_rates'].values()))
units = dict.fromkeys(outputs, mechanism['model']['kinetics']['rate_unit'])


In [ ]:
# Fit rates on a declared synthetic grid, never on desired trajectories.
axes = [np.linspace(*bounds, settings['training_points_per_axis']) for bounds in settings['validity_domain'].values()]
points = np.array(list(itertools.product(*axes)))
targets = []
for point in points:
    snapshot = SimpleNamespace(concentrations_mmol_l={s['name']: v for s,v in zip(settings['inputs'].values(), point)}, volume_l=1.)
    result = baseline.mechanism.rate_provider.evaluate(snapshot, {})
    targets.append([result.rates[name] for name in outputs])
targets = np.array(targets)
providers = {}
for degree, label in [(1, 'affine'), (2, 'polynomial')]:
    powers = [p for p in itertools.product(range(degree+1), repeat=len(axes)) if sum(p)<=degree]
    matrix = np.prod(points[:,None,:] ** np.array(powers)[None,:,:], axis=2)
    coefficients = np.linalg.lstsq(matrix, targets, rcond=None)[0]
    common = dict(identifier=label, version='1', inputs=settings['inputs'],
                  validity_domain=settings['validity_domain'], extrapolation='error')
    if degree == 1:
        fitted = {name: dict(unit=units[name], intercept=float(coefficients[powers.index((0,0)),i]),
            coefficients={alias:float(coefficients[powers.index(tuple(int(j==k) for j in range(len(axes)))),i])
                          for k,alias in enumerate(settings['inputs'])}) for i,name in enumerate(outputs)}
        providers[label] = dict(common, type='affine-surrogate', outputs=fitted)
    else:
        fitted = {name:dict(unit=units[name], coefficients=coefficients[:,i].tolist()) for i,name in enumerate(outputs)}
        providers[label] = dict(common, type='example-polynomial', powers=powers, outputs=fitted)
# Re-running this cell in a notebook does not re-register the same plugin.
if not globals().get('_polynomial_registered', False):
    register_rate_provider('example-polynomial', PolynomialProvider)
    _polynomial_registered = True


In [ ]:
assemblies = {'mechanistic': baseline}
for label, provider in providers.items():
    configured = deepcopy(mechanism)
    configured['rate_provider'] = provider
    assembly = build_bioreactor(case, configured, thermo)
    histories[label] = assembly.solve(verbose=False)
    assemblies[label] = assembly
def table(assembly, segments):
    names = list(assembly.phase.name_species)
    rows = []
    for segment in segments:
        volumes = np.asarray(segment.working_volume_liquid0).reshape(-1)*1000.
        for time, mass, volume in zip(segment.time, segment.mass_j_liquid0, volumes):
            concentrations = {name:mass[names.index(name)]*1000./volume for name in names}
            rows.append([time/86400., concentrations['glucose'],
                         concentrations['viable']/settings['cell_dry_mass_g']/1e9,
                         concentrations['dead']/settings['cell_dry_mass_g']/1e9,
                         concentrations['product'], volume])
    return np.array(rows)
tables = {label:table(assemblies[label], values) for label,values in histories.items()}
reference = tables['mechanistic']
report = {'scope':'Synthetic surrogate qualification, not experimental validation', 'runs':{}}
for label in providers:
    error = np.max(abs(tables[label][:,1:]-reference[:,1:])/np.maximum(np.max(abs(reference[:,1:]),axis=0),1e-30))
    # Check the declared domain over every exported state, not only the final one.
    species = list(assemblies[label].phase.name_species)
    for segment in histories[label]:
        for mass,vol in zip(segment.mass_j_liquid0, np.asarray(segment.working_volume_liquid0).reshape(-1)):
            snapshot = SimpleNamespace(concentrations_mmol_l={n:m/mw*1000./vol for n,m,mw in zip(species,mass,assemblies[label].phase.mw)},volume_l=vol*1000.)
            assert assemblies[label].mechanism.rate_provider.evaluate(snapshot,{}).inside_validity_domain
    report['runs'][label] = {'maximum_normalized_trajectory_error':float(error),'inside_validity_domain':True}
    assert error < settings['maximum_normalized_trajectory_error']
report


In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
labels = ['time_day','glucose_g_L','viable_1e6_cells_mL','dead_1e6_cells_mL','product_g_L','volume_L']
fig, axes = plt.subplots(2,2,figsize=(10,7),constrained_layout=True)
for label,values in tables.items():
    np.savetxt(EXPORT_DIR/f'{label}.csv',values,delimiter=',',header=','.join(labels),comments='')
    for ax,index in zip(axes.flat,[1,2,4,5]):
        ax.plot(values[:,0],values[:,index],label=label)
        ax.set(xlabel='Time (day)',ylabel=labels[index]); ax.legend()
for extension in ('png','svg'):
    fig.savefig(EXPORT_DIR/f'trajectories.{extension}')
plt.close(fig)
(EXPORT_DIR/'qualification.json').write_text(json.dumps(report,indent=2)+'\n')
for label,provider in providers.items():
    configured = deepcopy(mechanism); configured['rate_provider'] = provider
    (EXPORT_DIR/f'{label}_mechanism.json').write_text(json.dumps(configured,indent=2)+'\n')
